# CorrDiff-Mini smoke test (ERA5 → CARRA2)

Headless end-to-end **plumbing** check on a Kaggle GPU: dataset adapter → GPU LR conditioner
→ regression → diffusion → generate. Tiny data, a few dozen steps — this proves the pipeline
**runs**, not that it learns anything.

**Before running:**
1. Attach your uploaded smoke dataset (the trimmed `shard_2011_smoke.zarr`) to this kernel
   (Add Input → your dataset). It mounts under `/kaggle/input/...` and is auto-located below.
2. Make sure the repo/branch in the **Parameters** cell is pushed & reachable.
3. **Accelerator = `GPU T4 x2`** — NOT P100. physicsnemo needs torch≥2.10, and torch 2.10
   dropped Tesla P100 (sm_60) support, so a P100 will fail fast in the GPU-check cell.
   An API push defaults to P100, so set T4 explicitly (`--accelerator NvidiaTeslaT4` if your
   kaggle CLI supports it, otherwise once in the kernel's Settings).


## Parameters — edit these

In [ ]:
# ==== EDIT THESE ====
REPO_URL = "https://github.com/iolmoslau/era5-carra2-downscaling-canadian-arctic.git"
BRANCH   = "corrdiff-mini-training"
INPUT_ROOT = "/kaggle/input"          # Kaggle mounts attached datasets here

# Smoke-test size (tiny on purpose)
TRAIN_DURATION = 2000                 # processed samples per stage
TOTAL_BATCH    = 2
BATCH_PER_GPU  = 1
FP_OPT         = "fp32"               # simplest for a smoke test (T4 has fp16 but no bf16)
GEN_TIMES      = ["2011-01-01T00:00:00", "2011-01-05T12:00:00"]  # must exist in the shard

# physicsnemo requires torch>=2.10, which the Kaggle image already ships -- so we do NOT
# reinstall torch. Pin here only if a specific physicsnemo release is needed.
PHYSICSNEMO_SPEC = "nvidia-physicsnemo"   # e.g. "nvidia-physicsnemo==1.0.0"


## 1. Install physicsnemo — **before** importing torch

physicsnemo needs `torch>=2.10`, which Kaggle already provides, so we install only physicsnemo
+ a few light deps and **never touch torch** (reinstalling/reloading torch mid-kernel crashes it).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", PHYSICSNEMO_SPEC], check=False)
# Light deps our code needs that the Kaggle image lacks. The shard is zarr v3 -> zarr>=3.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "hydra-core>=1.2", "omegaconf>=2.3", "nvtx", "cftime>=1.6", "numba",
                "zarr>=3", "dask", "netCDF4", "wandb"], check=False)
print("installs complete")


## 2. GPU check (fail fast if we got a P100)

In [ ]:
import torch
print("torch", torch.__version__, "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU visible -- set the kernel Accelerator to 'GPU T4 x2'."
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"device: {name} | compute capability sm_{cap[0]}{cap[1]}")
if cap[0] < 7:
    raise SystemExit(
        f"{name} (sm_{cap[0]}{cap[1]}) is too old for torch {torch.__version__} (needs sm_70+). "
        "physicsnemo requires torch>=2.10, which dropped Pascal/P100. "
        "FIX: set the kernel Accelerator to 'GPU T4 x2' (T4 = sm_75) and re-run."
    )
try:
    import physicsnemo
    print("physicsnemo", getattr(physicsnemo, "__version__", "?"))
except Exception as e:
    print("physicsnemo import FAILED:", repr(e))


## 3. Get the code

In [ ]:
import os
# Clone OUTSIDE /kaggle/working so the repo is NOT saved into the kernel output.
# Only /kaggle/working is persisted, so `kaggle kernels output` stays small (log +
# rendered notebook + anything small we deliberately write to /kaggle/working).
CODE_DIR = "/tmp/corrdiff_code"
subprocess.run(["rm", "-rf", CODE_DIR], check=False)
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CODE_DIR])
assert r.returncode == 0, "git clone failed -- is the repo public and the branch pushed?"
TRAIN_DIR = os.path.join(CODE_DIR, "training_mini")
os.chdir(TRAIN_DIR)
print("cwd:", os.getcwd())
print(sorted(os.listdir(".")))


## 4. Locate the smoke shard + a small helper to run CLI steps

In [ ]:
import glob, json as _json

def _print_input_tree(root, max_dirs=80):
    print("=== /kaggle/input tree (truncated) ===")
    n = 0
    for dp, dns, fns in os.walk(root):
        depth = dp[len(root):].count(os.sep)
        print("  " * depth + (os.path.basename(dp) or dp) + "/")
        for f in sorted(fns)[:8]:
            print("  " * (depth + 1) + f)
        n += 1
        if n > max_dirs:
            print("... (truncated)"); break

def find_zarr_store_root(root):
    """Top-level zarr store dir, found by CONTENT (v3 group zarr.json or v2 .zgroup),
    regardless of the folder name. Kaggle sometimes drops the wrapping .zarr dir name."""
    for dp, dns, fns in os.walk(root):   # top-down: first hit is the outermost store
        if "zarr.json" in fns:
            try:
                if _json.load(open(os.path.join(dp, "zarr.json"))).get("node_type") == "group":
                    return dp
            except Exception:
                pass
        if ".zgroup" in fns:
            return dp
    return None

store = find_zarr_store_root(INPUT_ROOT)
if store is None:
    _print_input_tree(INPUT_ROOT)
    raise SystemExit("No zarr store found under /kaggle/input -- is the dataset attached AND complete?")

# make_stats.py / the adapter treat a single store by its `.zarr` suffix, so symlink if Kaggle
# stripped the name.
if store.endswith(".zarr"):
    SMOKE = store
else:
    SMOKE = "/tmp/smoke_store.zarr"
    subprocess.run(["rm", "-rf", SMOKE], check=False)
    os.symlink(store, SMOKE)
STATS = "/kaggle/working/stats_smoke.json"
print("store root found:", store)
print("SMOKE =", SMOKE)
print("store contents:", sorted(os.listdir(SMOKE)))

def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise SystemExit(f"command failed ({p.returncode}): {cmd}")

def latest_mdlus(keyword):
    ms = glob.glob(TRAIN_DIR + "/**/*.mdlus", recursive=True)
    hit = [m for m in ms if keyword in m.lower()]
    if not hit:
        raise SystemExit(f"no *.mdlus matching {keyword!r} found; all: {ms}")
    return sorted(hit, key=os.path.getmtime)[-1]


## 5. Train-only normalization stats

In [ ]:
sh(f"python tools/make_stats.py --stores '{SMOKE}' --out '{STATS}'")

## 6. Stage 1 — regression (mean predictor)

In [ ]:
reg_overrides = " ".join([
    f"++dataset.data_path='{SMOKE}'",
    f"++dataset.stats_path='{STATS}'",
    "'~validation'",
    f"++training.hp.training_duration={TRAIN_DURATION}",
    f"++training.hp.total_batch_size={TOTAL_BATCH}",
    f"++training.hp.batch_size_per_gpu={BATCH_PER_GPU}",
    f"++training.perf.fp_optimizations={FP_OPT}",
    "++training.perf.dataloader_workers=2",
    "++training.io.print_progress_freq=100",
    "++training.io.save_checkpoint_freq=1000",
])
sh(f"python train.py --config-name=config_training_era5_carra2_mini_regression {reg_overrides}")
REG_CKPT = latest_mdlus("regression")
print("REG_CKPT =", REG_CKPT)


## 7. Stage 2 — diffusion (residual predictor)

In [ ]:
diff_overrides = " ".join([
    f"++dataset.data_path='{SMOKE}'",
    f"++dataset.stats_path='{STATS}'",
    "'~validation'",
    f"++training.io.regression_checkpoint_path='{REG_CKPT}'",
    f"++training.hp.training_duration={TRAIN_DURATION}",
    f"++training.hp.total_batch_size={TOTAL_BATCH}",
    f"++training.hp.batch_size_per_gpu={BATCH_PER_GPU}",
    f"++training.perf.fp_optimizations={FP_OPT}",
    "++training.perf.dataloader_workers=2",
    "++training.io.print_progress_freq=100",
    "++training.io.save_checkpoint_freq=1000",
])
sh(f"python train.py --config-name=config_training_era5_carra2_mini_diffusion {diff_overrides}")
RES_CKPT = latest_mdlus("diffusion")
print("RES_CKPT =", RES_CKPT)


## 8. Generate on a couple of timestamps

In [ ]:
gen_times = "[" + ",".join(GEN_TIMES) + "]"
gen_overrides = " ".join([
    f"++dataset.data_path='{SMOKE}'",
    f"++dataset.stats_path='{STATS}'",
    f"++generation.io.reg_ckpt_filename='{REG_CKPT}'",
    f"++generation.io.res_ckpt_filename='{RES_CKPT}'",
    f"'++generation.times={gen_times}'",
])
sh(f"python generate.py --config-name=config_generate_era5_carra2_mini {gen_overrides}")


## 9. Peek at the output NetCDF

In [ ]:
import shutil, xarray as xr, numpy as np, matplotlib.pyplot as plt

ncs = sorted(glob.glob(TRAIN_DIR + "/**/*.nc", recursive=True), key=os.path.getmtime)
print("NetCDF outputs:", ncs)
nc = ncs[-1]

# Persist artifacts into /kaggle/working so `kaggle kernels output` downloads them
shutil.copy(nc, "/kaggle/working/corrdiff_output.nc")
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
for m in glob.glob(TRAIN_DIR + "/**/*.mdlus", recursive=True):
    shutil.copy(m, "/kaggle/working/checkpoints/" + os.path.basename(m))
print("saved to /kaggle/working:", sorted(os.listdir("/kaggle/working")))

pred = xr.open_dataset(nc, group="prediction")
truth = xr.open_dataset(nc, group="truth")
var = list(truth.data_vars)[0]
print("plotting channel:", var, "| pred sizes:", dict(pred[var].sizes))

p = np.asarray(pred[var].isel(ensemble=0, time=0))
t = np.asarray(truth[var].isel(time=0))
print(f"{var}: truth range [{t.min():.3g}, {t.max():.3g}] | pred range [{p.min():.3g}, {p.max():.3g}]")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(t, origin="lower"); ax[0].set_title(f"truth {var}")
ax[1].imshow(p, origin="lower"); ax[1].set_title(f"prediction {var}")
for a in ax: a.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/smoke_prediction.png", dpi=110, bbox_inches="tight")
plt.show()
print("SMOKE TEST OK: checkpoints + NetCDF produced and copied to /kaggle/working.")


## Notes / troubleshooting
- **P100 assigned again** → set Accelerator to `GPU T4 x2` (API pushes default to P100).
- **OOM on T4** → drop `TOTAL_BATCH`/`BATCH_PER_GPU` to 1; the 448×448 diffusion is the heavy part.
- **physicsnemo import/version error** → pin `PHYSICSNEMO_SPEC` and re-run (torch stays at Kaggle's 2.10).
- **`git clone` fails** → the branch must be pushed and the repo reachable (public, or add a token).
- **`times` not found** → `GEN_TIMES` must be 3-hourly stamps present in the trimmed shard (early Jan 2011).
